# BitRoss training (Colab Pro)

This notebook trains the CLIP pixel-DiT + rectified-flow model. The Cursor remote box has **no GPU**, so this is the intended training path.

**If the Train cell still shows `subprocess.check_call`, this notebook copy is stale.** Git pull updates `/content/BitRoss` on disk; it does **not** update the cells already open in Colab. Close this tab and reopen from GitHub:

https://colab.research.google.com/github/OVAWARE/BitRoss/blob/cursor/hq-cvae-architecture-5a9f/BitRoss_train.ipynb

You should see `Training code:` plus a SHA in the Train cell (frozen CLIP, not batch 512). Captions are **item names only** (no mod slugs). After a CUDA OOM, use **Runtime → Restart session** before re-running Train — leftover VRAM from the crashed process will make the next attempt fail too.

## GPU (do this first)

**Runtime → Change runtime type**

| GPU | Use it? | Why |
|---|---|---|
| **L4** (default) | **Yes** | Best price:quality. Ada Lovelace, bf16, ~1.7 CU/hr. This 16×16 DiT is tiny; L4 is plenty and much faster than a T4. |
| **T4** | Fine | Cheapest (~1.2 CU/hr). fp16 instead of bf16. Use if L4 is unavailable. |
| **A100 / H100 / G4** | **No** | 3–7× the compute-unit burn, almost no speedup on this model. |

Colab Pro sessions last up to 24h. Checkpoints go to Google Drive so a disconnect is not a wipe. Re-run from the train cell with `RESUME = "auto"`.

## Hugging Face dataset

Gated dump: [OVAWARE/16xModdedMinecraft](https://huggingface.co/datasets/OVAWARE/16xModdedMinecraft) (~1.03M 16×16 RGBA rows: `image`, `file_name`, `type`, `mod_slug`, … — **no captions**).

1. Accept the terms on that page while logged in
2. Paste your Hugging Face token into `HF_TOKEN` in the prepare cell (or add a Colab secret named `HF_TOKEN`)

Do **not** commit the token. The prepare cell keeps `type=item` sprites with mixed alpha (drops ≥95% transparent specks — fewer than ~13 of 256 pixels — and ≥99% opaque tiles) and writes a Drive cache (`images_u8.npy` + captions). Training also re-applies that 5% opaque floor when it loads the pack, so an older Drive cache does not need a full rebuild. Captions are the formatted item filename only — never the mod name — so you can prompt `pixel art minecraft item, diamond sword`. Training copies that cache off Drive onto local disk first. Checkpoints are written atomically as `BitRoss_latest.pth` (resume) plus two rotated epoch files so Drive does not fill up.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → GPU → L4 (or T4)."
)
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name}  CC {cap}  VRAM {vram:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")

expensive = any(k in name.upper() for k in ("A100", "H100", "A800", "PRO 6000", "G4"))
if expensive:
    raise SystemExit(
        f"{name} burns Colab compute units for almost no gain on BitRoss.\n"
        "Runtime → Change runtime type → L4 (preferred) or T4."
    )
if "L4" in name:
    print("L4: best price/quality for this run.")
elif "T4" in name:
    print("T4: cheapest option, slightly slower. Fine for 16x16.")
else:
    print("Unknown GPU — continuing, but L4 is the intended Pro default.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/BitRoss"
!mkdir -p "{DRIVE_ROOT}/models" "{DRIVE_ROOT}/data"

In [ ]:
# @title Repo + deps
BRANCH = "cursor/hq-cvae-architecture-5a9f"  # @param {type:"string"}
REPO = "https://github.com/OVAWARE/BitRoss.git"

import os
os.chdir("/content")
if not os.path.isdir("/content/BitRoss/.git"):
    !git clone --branch {BRANCH} --depth 50 {REPO}
else:
    os.chdir("/content/BitRoss")
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
os.chdir("/content/BitRoss")
!git log -1 --oneline
print("Need a SHA from Aug 15+ (frozen CLIP). If git log shows CoOp-in-CLIP / batch 512, re-run this cell.")
# Colab already ships CUDA torch — do not pip-install the CPU wheel.
# Pin transformers: CLIP CoOp encoding depends on this API.
%pip install -q "transformers==5.15.0" wandb pillow datasets huggingface_hub

In [ ]:
# @title Prepare Hugging Face dataset (cached on Drive)
HF_TOKEN = ""  # @param {type:"string"}
FORCE_REBUILD = False  # @param {type:"boolean"}
MAX_SAMPLES = 0  # @param {type:"integer"}

import os
from pathlib import Path

os.chdir("/content/BitRoss")
PROCESSED = "/content/drive/MyDrive/BitRoss/data/processed-items"
os.makedirs(os.path.dirname(PROCESSED), exist_ok=True)

# Token: paste in the form above, else Colab secret HF_TOKEN, else env.
# Never commit a real token — this notebook is public on GitHub.
token = (HF_TOKEN or "").strip() or os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass
if not token:
    raise SystemExit(
        "Missing HF_TOKEN. Paste it in the form field above (or add a Colab secret), "
        "and accept the dataset terms at "
        "https://huggingface.co/datasets/OVAWARE/16xModdedMinecraft"
    )
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

from huggingface_hub import login
login(token=token)

marker = Path(PROCESSED) / "prepare_stats.json"
if marker.exists() and not FORCE_REBUILD:
    print("Using cached processed dataset:", PROCESSED)
    print(marker.read_text()[:1500])
else:
    extra = []
    if MAX_SAMPLES and MAX_SAMPLES > 0:
        extra += ["--max_samples", str(MAX_SAMPLES)]
    import subprocess, sys
    cmd = [
        sys.executable, "prepare_dataset.py",
        "--out_dir", PROCESSED,
        "--num_proc", "2",
        "--hf_token", token,
    ] + extra
    print("prepare_dataset.py --out_dir", PROCESSED, "--num_proc 2")
    subprocess.check_call(cmd)


In [ ]:
# @title Train
EPOCHS = 800  # @param {type:"integer"}
BATCH_SIZE = 0  # @param {type:"integer"}
RESUME = "auto"  # @param ["auto", ""]
USE_WANDB = False  # @param {type:"boolean"}
WANDB_API_KEY = ""  # @param {type:"string"}
BRANCH = "cursor/hq-cvae-architecture-5a9f"  # @param {type:"string"}

import os, sys

os.chdir("/content/BitRoss")
# Always sync this cell to GitHub so Colab cannot train a stale clone.
# Shallow clones + a cached `import model` from an earlier run are how the
# L4 OOM'd at batch 512 on CoOp-through-CLIP after the freeze was already pushed.
import subprocess
subprocess.check_call(["git", "fetch", "origin", BRANCH])
subprocess.check_call(["git", "checkout", BRANCH])
subprocess.check_call(["git", "reset", "--hard", f"origin/{BRANCH}"])
print("Training code:", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())
for name in list(sys.modules):
    if name in {"model", "train", "prepare_dataset", "generate"} or name.startswith(
        ("model.", "prepare_dataset.")
    ):
        del sys.modules[name]

SAVE_DIR = "/content/drive/MyDrive/BitRoss/models"
os.makedirs(SAVE_DIR, exist_ok=True)

if USE_WANDB:
    if WANDB_API_KEY:
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    import wandb
    wandb.login()

# Run in-process so a crash shows the real train.py traceback (not CalledProcessError).
argv = [
    "train.py",
    "--processed_dir", "/content/drive/MyDrive/BitRoss/data/processed-items",
    "--save_dir", SAVE_DIR,
    "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", "0",
    "--keep_checkpoints", "2",
]
if RESUME:
    argv += ["--resume", RESUME]
if not USE_WANDB:
    argv.append("--no_wandb")
print(" ".join([sys.executable] + argv))
sys.argv = argv
import runpy
runpy.run_path("train.py", run_name="__main__")

In [ ]:
# @title Sample from the latest checkpoint
PROMPT = "pixel art minecraft item, diamond sword, blue crystal blade"  # @param {type:"string"}
CFG_SCALE = 2.5  # @param {type:"number"}
STEPS = 20  # @param {type:"integer"}

import os, glob
from IPython.display import display
from PIL import Image

os.chdir("/content/BitRoss")
SAVE_DIR = "/content/drive/MyDrive/BitRoss/models"
latest = os.path.join(SAVE_DIR, "BitRoss_latest.pth")
ckpts = [latest] if os.path.isfile(latest) else sorted(glob.glob(SAVE_DIR + "/*.pth"), key=os.path.getmtime)
assert ckpts, f"No checkpoints in {SAVE_DIR}"
ckpt = ckpts[-1] if not os.path.isfile(latest) else latest
out = "/content/sample.png"
print("Using", ckpt)
!python generate.py --model_path "{ckpt}" --prompt "{PROMPT}" --output "{out}" --cfg_scale {CFG_SCALE} --steps {STEPS} --size 256
display(Image.open(out))